# Session 3 — zero-structure EDA + first scored baselines

1. Per-series zero table (mean_sales, zero_rate, weeks) — promoted from `01_practice.ipynb`.
2. Dead-series check (`zero_rate == 1.0`) — CONTEXT.md says this should be 0.
3. Zero-position classification: leading, interior, trailing, always-zero.
4. Score MA12 on the 13-week holdout with the official metric, with a component breakdown.
5. Compare MA12 against a small suite of other naive baselines.

In [1]:
import os
from pathlib import Path

# Make this notebook robust to being launched from the repo root or from
# notebooks/ — walk up until we find the repo marker, then work from there.
# Guard against reaching the filesystem root without finding it (parent of
# root is root itself, so an unguarded loop would spin forever).
current = Path.cwd()
while not (current / "CONTEXT.md").is_file():
    if current == current.parent:
        raise FileNotFoundError("Could not find CONTEXT.md in any parent directory")
    current = current.parent
os.chdir(current)
print("repo root:", Path.cwd())

import pandas as pd

from src.data import KEY
from src.backtest import (
    split_last_horizon,
    ma12_forecast,
    zero_forecast,
    seasonal_naive_forecast,
    score_forecast,
    HORIZON,
)
from src.metrics import vn1_score, vn1_score_breakdown

df = pd.read_parquet("data/processed/vn1_long.parquet")
df.shape

repo root: D:\vn1-forecasting


(2559010, 7)

## 1. Per-series zero table

In [2]:
# Cell — per-series zero table
zero_table = df.groupby(KEY)["sales"].agg(
    mean_sales="mean",
    zero_rate=lambda s: (s == 0).mean(),
    weeks="count",
)
print("series:", len(zero_table))
zero_table.describe().round(3)

series: 15053


,mean_sales,zero_rate,weeks
count,15053.000,15053.000,15053.0
mean,12.684,0.718,170.0
std,82.230,0.297,0.0
min,0.006,0.000,170.0
25%,0.071,0.547,170.0
50%,0.559,0.835,170.0
75%,4.235,0.959,170.0
max,4808.841,0.994,170.0


In [3]:
# Cell — distribution of zero_rate across all series
# (bucketed counts, not a plot — matplotlib patch rendering is broken in this
# env right now; see PROGRESS.md note. Revisit once that's fixed.)
bins = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ["<=10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%"]
bucketed = pd.cut(zero_table["zero_rate"], bins=bins, labels=labels, include_lowest=True)
print(bucketed.value_counts().sort_index())

print()
print("mean zero_rate  :", round(zero_table["zero_rate"].mean(), 3))
print("median zero_rate:", round(zero_table["zero_rate"].median(), 3))
print(">90% zero weeks :", (zero_table["zero_rate"] > 0.90).sum())
print("<10% zero weeks :", (zero_table["zero_rate"] < 0.10).sum())
print("max zero_rate   :", zero_table["zero_rate"].max())
# Note: "<=10%" (pd.cut, right-inclusive) and "<10% zero weeks" above differ
# by design — pd.cut's first bin includes series sitting exactly at 0.10.

zero_rate
<=10%      1021
10-30%     1007
30-50%     1398
50-70%     1919
70-90%     3694
90-100%    6014
Name: count, dtype: int64

mean zero_rate  : 0.718
median zero_rate: 0.835
>90% zero weeks : 6014
<10% zero weeks : 994
max zero_rate   : 0.9941176470588236


## 2. Dead-series check

In [4]:
# Cell — dead series: zero_rate == 1.0 (never sold at all)
dead = zero_table[zero_table["zero_rate"] == 1.0]
print("dead series (zero_rate == 1.0):", len(dead))
# Expect 0 — CONTEXT.md: every series has at least one positive sale.

dead series (zero_rate == 1.0): 0


## 3. Zero-position classification

For each series, find the first and last week with a positive sale, then
classify every zero-sales week by its position relative to those dates:

- **leading**  — zero-sales weeks *before* the first observed positive sale
- **interior** — zero-sales weeks *between* the first and last positive sale
- **trailing** — zero-sales weeks *after* the last observed positive sale
- **always-zero** — a series with no positive sale at all (none exist here)

This is purely positional — deliberately not called "lifecycle," since that
word implies launch/growth/decline causes we can't actually see. VN1 has no
launch-date, discontinuation, or inventory/availability data, so we cannot
claim these zeros mean "pre-launch," "discontinued," or "genuine
intermittency" — a zero could equally be real zero demand or demand
constrained by a stockout we can't see (CONTEXT.md's demand-vs-sales
distinction). Preserve the positions; don't infer the cause.

In [5]:
# Cell — first/last positive-sale week per series
df_sorted = df.sort_values(KEY + ["date"])
positive = df_sorted[df_sorted["sales"] > 0]

first_last = positive.groupby(KEY)["date"].agg(first_sale="min", last_sale="max")
positions = df_sorted.merge(first_last, on=KEY, how="left")

# A series with no positive sale at all has no entry in first_last, so
# first_sale/last_sale come back NaT after the left merge. Gate on that
# explicitly instead of relying on "not leading and not trailing" — NaT
# comparisons are always False, so without this gate every zero of an
# always-zero series would silently fall into "interior".
is_zero = positions["sales"] == 0
always_zero = positions["first_sale"].isna()

leading = is_zero & ~always_zero & (positions["date"] < positions["first_sale"])
trailing = is_zero & ~always_zero & (positions["date"] > positions["last_sale"])
interior = is_zero & ~always_zero & ~leading & ~trailing

counts = pd.Series({
    "leading": leading.sum(),
    "interior": interior.sum(),
    "trailing": trailing.sum(),
    "always-zero": (is_zero & always_zero).sum(),
})
counts

leading        1139262
interior        467146
trailing        230029
always-zero          0
dtype: int64

## 4. MA12 baseline, scored on the 13-week holdout

In [6]:
# Cell — hold out the last 13 weeks, forecast with MA12, score
train, test = split_last_horizon(df, horizon=HORIZON)
print("train dates:", train["date"].min(), "->", train["date"].max())
print("test dates :", test["date"].min(), "->", test["date"].max())
print("train weeks:", train["date"].nunique(), " test weeks:", test["date"].nunique())

train dates: 2020-07-06 00:00:00 -> 2023-07-03 00:00:00
test dates : 2023-07-10 00:00:00 -> 2023-10-02 00:00:00
train weeks: 157  test weeks: 13


In [7]:
# Cell — build forecast, align to test rows, score (with component breakdown)
forecast = ma12_forecast(train)
scored = score_forecast(test, forecast)

breakdown = vn1_score_breakdown(scored["sales"], scored["forecast"])
for k, v in breakdown.items():
    print(f"{k:15s}: {v:,.4f}" if k != "score" else f"{k:15s}: {v:.4f}")

total_actual   : 3,508,827.0000
total_predicted: 3,354,248.0833
signed_bias    : 154,578.9167
abs_error_sum  : 1,657,059.5833
bias_term      : 154,578.9167
score          : 0.5163


## 5. Baseline suite

A single benchmark (MA12) doesn't say much on its own. Compare it against a
spread of cheap naive alternatives on the same holdout, using the same
`score_forecast` + `vn1_score_breakdown` path, so a LightGBM model later has
more than one number to actually beat.

- **zero** — always forecast 0
- **naive-last** — repeat the most recent observed week (MA with window=1)
- **MA4 / MA8 / MA12 / MA26** — moving averages of different lengths
- **seasonal-naive-52** — repeat the value from 52 weeks earlier (captures
  annual seasonality, if any; train has 157 weeks so this is well-supported)

In [8]:
# Cell — score a suite of naive baselines on the same holdout
results = {}

scored_zero = score_forecast(test, zero_forecast(train))
results["zero"] = vn1_score(scored_zero["sales"], scored_zero["forecast"])

for name, window in [("naive-last", 1), ("MA4", 4), ("MA8", 8), ("MA12", 12), ("MA26", 26)]:
    scored_b = score_forecast(test, ma12_forecast(train, window=window))
    results[name] = vn1_score(scored_b["sales"], scored_b["forecast"])

scored_seasonal = score_forecast(test, seasonal_naive_forecast(df, test, lag_weeks=52))
results["seasonal-naive-52"] = vn1_score(scored_seasonal["sales"], scored_seasonal["forecast"])

suite = pd.Series(results, name="vn1_score").sort_values()
suite

MA4                  0.498708
MA8                  0.511414
MA12                 0.516309
naive-last           0.524666
MA26                 0.561836
seasonal-naive-52    1.049790
zero                 2.000000
Name: vn1_score, dtype: float64